# Off-Policy Continuous Control: DDPG -> TD3 -> SAC on Pendulum-v1

*A sibling to the discrete-action RL companion notebook. Audience: AI/ML PhD.*

This notebook puts the three canonical **off-policy actor-critic** algorithms for
**continuous** control head-to-head on the classic `Pendulum-v1` swing-up task:

| Algo | Actor | Critic(s) | Exploration | Key idea |
|------|-------|-----------|-------------|----------|
| **DDPG** | deterministic $\mu(s)$ | single $Q(s,a)$ | additive Gaussian noise | DPG + replay + target nets |
| **TD3**  | deterministic $\mu(s)$ | **twin** $Q_{1},Q_{2}$ (min) | additive Gaussian noise | fixes DDPG's overestimation with 3 tricks |
| **SAC**  | **stochastic** squashed-Gaussian | twin $Q_{1},Q_{2}$ (min) | **entropy-regularised** policy | max-entropy RL, reparameterized |

**Everything below is shared** -- the replay buffer, target networks, Polyak
averaging, network sizes, optimizer, eval protocol -- so the *only* moving part is
the **algorithm**. That makes the comparison clean and the pedagogy crisp:

> **DDPG** tends to *overestimate* $Q$ (maximization bias) and is brittle. **TD3**'s
> three tricks (twin critics, delayed policy updates, target-policy smoothing) tame
> that bias. **SAC** adds an *entropy bonus* that yields strong, stable exploration
> and is usually the most robust of the three.

Watch for this in the learning curves: DDPG is often noisier / can stall; TD3 is
steadier; SAC typically climbs fastest and most reliably.

### Runtime note
This runs on **free Colab CPU** in a few minutes -- but it is **slower than the
discrete-action notebooks** (more gradient steps, two networks). If you are
impatient, drop `TOTAL_STEPS` (e.g. to 8000) in the config cell; you will still
see the qualitative ordering, just with rougher curves.


In [ ]:
%pip install -q "gymnasium[classic-control]>=0.29" "pygame>=2.5"


In [ ]:
# ---- imports -------------------------------------------------------------
import time, math, copy, random
from dataclasses import dataclass, field
from typing import List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import gymnasium as gym
import matplotlib
import matplotlib.pyplot as plt

# CPU only -- this notebook is sized for free Colab without a GPU.
DEVICE = torch.device("cpu")
torch.set_num_threads(max(1, (torch.get_num_threads() or 1)))

print("torch      :", torch.__version__)
print("gymnasium  :", gym.__version__)
print("numpy      :", np.__version__)
print("matplotlib :", matplotlib.__version__)
print("device     :", DEVICE)


In [ ]:
# ===========================================================================
# GLOBAL CONFIG  -- the single source of truth for every algorithm.
# Tweak TOTAL_STEPS down if you are impatient (see intro).
# ===========================================================================
ENV_ID      = "Pendulum-v1"

GAMMA       = 0.99      # discount factor
TAU         = 0.005     # Polyak averaging coefficient for target nets (soft update)

TOTAL_STEPS = 15000     # env steps PER (algo, seed) run. 15k is modest but enough.
START_STEPS = 1000      # warmup: pure random actions to seed the replay buffer
EVAL_EVERY  = 1000      # run a (deterministic) evaluation every this many env steps
EVAL_EPISODES = 5       # episodes averaged per evaluation point

BATCH_SIZE  = 128
BUFFER_SIZE = 100_000
HIDDEN      = 128       # width of the 2-hidden-layer MLPs (2 x 128)

LR_ACTOR    = 3e-4
LR_CRITIC   = 3e-4

SEEDS       = [0, 1]    # 2 seeds keeps the run fast while still showing mean +/- std

# ---- DDPG / TD3 exploration + TD3 tricks --------------------------------
ACT_NOISE     = 0.1     # std of Gaussian exploration noise (in ACTION units)
POLICY_DELAY  = 2       # TD3: update actor (and targets) every k critic updates
TARGET_NOISE  = 0.2     # TD3: std of smoothing noise added to target action
NOISE_CLIP    = 0.5     # TD3: clip range for the target smoothing noise

# ---- SAC ----------------------------------------------------------------
SAC_ALPHA   = 0.2       # fixed entropy temperature (auto-tuning left as an exercise)

# Pendulum specifics (filled in once we make the env, but stated here for clarity):
#   obs dim = 3   ([cos t, sin t, theta_dot])
#   act dim = 1   (torque), bounds [-2, +2]
print("Config loaded. TOTAL_STEPS =", TOTAL_STEPS, "| SEEDS =", SEEDS)


## The task: `Pendulum-v1`

A frictionless pendulum starts in a random position. You apply a **torque** and try
to **swing it up and balance** it pointing straight up, using as little force as
possible.

- **Observation** (3-dim): $[\cos\theta,\ \sin\theta,\ \dot\theta]$ -- the angle is given
  as its cosine/sine (so the network never sees a discontinuity at $\pm\pi$) plus the
  angular velocity.
- **Action** (1-dim, *continuous*): torque $\in [-2, +2]$.
- **Reward** (always $\le 0$): $-(\theta^2 + 0.1\,\dot\theta^2 + 0.001\,a^2)$ -- you pay
  for being off-vertical, for spinning, and for using torque. The best you can do is
  $0$ (perfectly balanced, still, no torque); a good policy reaches roughly
  $\ge -200$ per episode (200 steps).

Because the reward is dense and the action is a single bounded scalar, Pendulum is the
perfect minimal benchmark for continuous-control algorithms.

Let's first watch a **random agent** flail around.


In [ ]:
# ---- helper: render an episode to an inline GIF ---------------------------
import io, base64
from IPython.display import HTML

def episode_frames(env, act_fn, max_steps=200, seed=0):
    """Roll out one episode, collecting rgb frames. act_fn(obs)->action(np)."""
    obs, _ = env.reset(seed=seed)
    frames = []
    for _ in range(max_steps):
        f = env.render()
        if f is not None:
            frames.append(np.asarray(f))
        a = act_fn(obs)
        obs, r, term, trunc, _ = env.step(a)
        if term or trunc:
            break
    return frames

def frames_to_gif_html(frames, fps=30, width=320):
    """Encode a list of HxWx3 uint8 frames as an inline animated GIF."""
    if not frames:
        return HTML("<i>(no frames rendered)</i>")
    try:
        import imageio.v2 as imageio
    except Exception:
        import imageio
    buf = io.BytesIO()
    imageio.mimsave(buf, frames, format="GIF", fps=fps, loop=0)
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    return HTML(f'<img src="data:image/gif;base64,{b64}" width="{width}"/>')

# Random agent: sample uniformly from the action space.
_env = gym.make(ENV_ID, render_mode="rgb_array")
_rand_act = lambda obs: _env.action_space.sample()
_frames = episode_frames(_env, _rand_act, max_steps=200, seed=0)
_env.close()
print("rendered", len(_frames), "frames")
frames_to_gif_html(_frames, fps=30)


## Shared component 1: the replay buffer

Off-policy methods learn from a buffer of past transitions
$(s, a, r, s', \text{done})$, sampled in random mini-batches. This decorrelates the
data and lets us reuse each transition many times. Plain NumPy ring buffer -- nothing
fancy.


In [ ]:
class ReplayBuffer:
    """Fixed-size NumPy ring buffer of transitions (s, a, r, s2, done)."""
    def __init__(self, obs_dim, act_dim, size):
        self.obs  = np.zeros((size, obs_dim), dtype=np.float32)
        self.obs2 = np.zeros((size, obs_dim), dtype=np.float32)
        self.act  = np.zeros((size, act_dim), dtype=np.float32)
        self.rew  = np.zeros((size, 1),       dtype=np.float32)
        self.done = np.zeros((size, 1),       dtype=np.float32)
        self.max_size = size
        self.ptr = 0
        self.n   = 0

    def add(self, s, a, r, s2, done):
        i = self.ptr
        self.obs[i]  = s
        self.act[i]  = a
        self.rew[i]  = r
        self.obs2[i] = s2
        self.done[i] = float(done)
        self.ptr = (self.ptr + 1) % self.max_size
        self.n   = min(self.n + 1, self.max_size)

    def sample(self, batch_size):
        idx = np.random.randint(0, self.n, size=batch_size)
        to_t = lambda x: torch.as_tensor(x[idx], device=DEVICE)
        return (to_t(self.obs), to_t(self.act), to_t(self.rew),
                to_t(self.obs2), to_t(self.done))

    def __len__(self):
        return self.n


## Shared component 2: the networks

We use small 2-hidden-layer MLPs (width `HIDDEN = 128`). Two actor flavours:

- **Deterministic actor** (DDPG, TD3): $\mu_\phi(s) \to a$, squashed with `tanh` and
  scaled to the action bounds.
- **Squashed-Gaussian actor** (SAC): outputs a mean and log-std; we sample with the
  **reparameterization trick** $a = \tanh(\mu + \sigma\,\epsilon)$ and apply the
  **tanh change-of-variables correction** to the log-probability (this is the part
  people get wrong -- see the comment).

Critics are plain $Q(s,a) \to \mathbb{R}$ MLPs. TD3 and SAC use **twin** critics.


In [ ]:
def mlp(sizes, activation=nn.ReLU, out_activation=nn.Identity):
    """Build a simple feed-forward MLP from a list of layer sizes."""
    layers = []
    for j in range(len(sizes) - 1):
        act = activation if j < len(sizes) - 2 else out_activation
        layers += [nn.Linear(sizes[j], sizes[j + 1]), act()]
    return nn.Sequential(*layers)


# ---------------------------------------------------------------------------
# Deterministic actor (DDPG / TD3): mu(s) -> action in [-act_limit, act_limit]
# ---------------------------------------------------------------------------
class DetActor(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden, act_limit):
        super().__init__()
        self.net = mlp([obs_dim, hidden, hidden, act_dim],
                       activation=nn.ReLU, out_activation=nn.Tanh)
        self.act_limit = float(act_limit)

    def forward(self, obs):
        # tanh -> (-1, 1), then scale to the action bounds.
        return self.act_limit * self.net(obs)


# ---------------------------------------------------------------------------
# Q-critic: Q(s, a) -> scalar
# ---------------------------------------------------------------------------
class QCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden):
        super().__init__()
        self.net = mlp([obs_dim + act_dim, hidden, hidden, 1],
                       activation=nn.ReLU, out_activation=nn.Identity)

    def forward(self, obs, act):
        return self.net(torch.cat([obs, act], dim=-1))  # (B, 1)


# ---------------------------------------------------------------------------
# Squashed-Gaussian actor (SAC).
#   - outputs mean and (clamped) log_std
#   - reparameterized sample:  u = mu + sigma * eps ;  a = tanh(u) * act_limit
#   - log prob gets the tanh Jacobian correction:
#       log p(a) = log N(u) - sum_i log(1 - tanh(u_i)^2)
#     We use the numerically-stable form  log(1 - tanh(u)^2) = 2*(log2 - u - softplus(-2u)).
# ---------------------------------------------------------------------------
LOG_STD_MIN, LOG_STD_MAX = -20.0, 2.0

class SquashedGaussianActor(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden, act_limit):
        super().__init__()
        self.body    = mlp([obs_dim, hidden, hidden], activation=nn.ReLU,
                           out_activation=nn.ReLU)
        self.mu_head     = nn.Linear(hidden, act_dim)
        self.log_std_head = nn.Linear(hidden, act_dim)
        self.act_limit   = float(act_limit)

    def forward(self, obs, deterministic=False, with_logprob=True):
        h = self.body(obs)
        mu = self.mu_head(h)
        log_std = self.log_std_head(h).clamp(LOG_STD_MIN, LOG_STD_MAX)
        std = torch.exp(log_std)

        if deterministic:
            u = mu                      # used at evaluation time
        else:
            eps = torch.randn_like(mu)  # reparameterization trick
            u = mu + std * eps

        if with_logprob:
            # log N(u | mu, std), summed over action dims
            logp = (-0.5 * (((u - mu) / (std + 1e-8)) ** 2)
                    - log_std - 0.5 * math.log(2 * math.pi)).sum(dim=-1, keepdim=True)
            # tanh correction (stable form), summed over action dims
            logp -= (2 * (math.log(2) - u - F.softplus(-2 * u))).sum(dim=-1, keepdim=True)
        else:
            logp = None

        a = torch.tanh(u) * self.act_limit
        return a, logp


## The one training function: `train(algo, seed)`

A single function implements all three algorithms; we branch on `algo` only where the
math genuinely differs. The skeleton is identical:

1. warm up with random actions (`START_STEPS`),
2. then act with the current policy (+ exploration noise for DDPG/TD3, stochastic
   sampling for SAC),
3. store the transition, sample a batch, update critic(s),
4. update the actor (DDPG every step; **TD3 every `POLICY_DELAY` steps**; SAC every step),
5. **Polyak**-average the target networks,
6. periodically run a deterministic evaluation and log the return.

The algorithm-specific bits:

- **DDPG target**: $y = r + \gamma (1-d)\, Q_{\text{targ}}(s', \mu_{\text{targ}}(s'))$.
- **TD3 target**: twin critics + **target-policy smoothing** (clipped noise on
  $\mu_{\text{targ}}(s')$) + **min** of the two target Q's; actor & targets updated only
  every `POLICY_DELAY` steps.
- **SAC target**: sample $a' \sim \pi(\cdot|s')$, use
  $y = r + \gamma(1-d)\,[\min_i Q_{\text{targ},i}(s',a') - \alpha \log\pi(a'|s')]$; the
  actor maximizes $\min_i Q_i(s,\tilde a) - \alpha \log\pi(\tilde a|s)$ with $\tilde a$
  reparameterized.


In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


def evaluate(actor, algo, act_limit, n_episodes=EVAL_EPISODES, seed=12345):
    """Deterministic greedy evaluation; returns mean episodic return."""
    eval_env = gym.make(ENV_ID)
    returns = []
    for ep in range(n_episodes):
        obs, _ = eval_env.reset(seed=seed + ep)
        done = False; ep_ret = 0.0
        while not done:
            o = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            with torch.no_grad():
                if algo == "SAC":
                    a, _ = actor(o, deterministic=True, with_logprob=False)
                    a = a.cpu().numpy()[0]
                else:
                    a = actor(o).cpu().numpy()[0]
            a = np.clip(a, -act_limit, act_limit)   # safety clip to env bounds
            obs, r, term, trunc, _ = eval_env.step(a)
            ep_ret += r
            done = term or trunc
        returns.append(ep_ret)
    eval_env.close()
    return float(np.mean(returns))


def train(algo, seed, total_steps=TOTAL_STEPS, verbose=True):
    """Train one of {'DDPG','TD3','SAC'} for `total_steps`; return (steps, curve, actor)."""
    assert algo in ("DDPG", "TD3", "SAC")
    set_seed(seed)
    t0 = time.time()

    env = gym.make(ENV_ID)
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]
    act_limit = float(env.action_space.high[0])   # Pendulum: 2.0

    buf = ReplayBuffer(obs_dim, act_dim, BUFFER_SIZE)

    # ---- build actor + critic(s) (+ targets) depending on algo -------------
    twin = (algo in ("TD3", "SAC"))
    if algo == "SAC":
        actor = SquashedGaussianActor(obs_dim, act_dim, HIDDEN, act_limit).to(DEVICE)
    else:
        actor = DetActor(obs_dim, act_dim, HIDDEN, act_limit).to(DEVICE)

    q1 = QCritic(obs_dim, act_dim, HIDDEN).to(DEVICE)
    q2 = QCritic(obs_dim, act_dim, HIDDEN).to(DEVICE) if twin else None

    actor_targ = copy.deepcopy(actor)             # DDPG/TD3 use a target actor;
    q1_targ = copy.deepcopy(q1)                   # SAC does not need a target actor
    q2_targ = copy.deepcopy(q2) if twin else None
    for p in actor_targ.parameters(): p.requires_grad_(False)
    for p in q1_targ.parameters():    p.requires_grad_(False)
    if twin:
        for p in q2_targ.parameters(): p.requires_grad_(False)

    q_params = list(q1.parameters()) + (list(q2.parameters()) if twin else [])
    pi_opt = torch.optim.Adam(actor.parameters(), lr=LR_ACTOR)
    q_opt  = torch.optim.Adam(q_params,           lr=LR_CRITIC)

    def polyak(target, source):
        with torch.no_grad():
            for pt, ps in zip(target.parameters(), source.parameters()):
                pt.mul_(1 - TAU).add_(TAU * ps)

    # ---- action selection during data collection ---------------------------
    def select_action(obs, step):
        if step < START_STEPS:
            return env.action_space.sample()              # pure exploration warmup
        o = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            if algo == "SAC":
                a, _ = actor(o, deterministic=False, with_logprob=False)
                a = a.cpu().numpy()[0]
            else:  # DDPG / TD3: deterministic action + Gaussian exploration noise
                a = actor(o).cpu().numpy()[0]
                a = a + ACT_NOISE * act_limit * np.random.randn(act_dim)
        return np.clip(a, -act_limit, act_limit)

    # ---- one gradient update ----------------------------------------------
    def update(step):
        s, a, r, s2, d = buf.sample(BATCH_SIZE)

        # ----- critic target -----
        with torch.no_grad():
            if algo == "DDPG":
                a2 = actor_targ(s2)
                q_targ = q1_targ(s2, a2)
            elif algo == "TD3":
                # target-policy smoothing: clipped noise on the target action
                noise = (torch.randn_like(a) * TARGET_NOISE).clamp(-NOISE_CLIP, NOISE_CLIP)
                a2 = (actor_targ(s2) + noise).clamp(-act_limit, act_limit)
                q_targ = torch.min(q1_targ(s2, a2), q2_targ(s2, a2))   # twin -> min
            else:  # SAC
                a2, logp2 = actor(s2, deterministic=False, with_logprob=True)
                q_targ = torch.min(q1_targ(s2, a2), q2_targ(s2, a2))
                q_targ = q_targ - SAC_ALPHA * logp2                    # entropy term
            y = r + GAMMA * (1.0 - d) * q_targ

        # ----- critic loss + step -----
        q1_loss = F.mse_loss(q1(s, a), y)
        q_loss = q1_loss + (F.mse_loss(q2(s, a), y) if twin else 0.0)
        q_opt.zero_grad(set_to_none=True)
        q_loss.backward()
        q_opt.step()

        # ----- actor update -----
        # DDPG/SAC: every step.  TD3: only every POLICY_DELAY steps (delayed updates).
        do_actor = (algo != "TD3") or (step % POLICY_DELAY == 0)
        if do_actor:
            for p in q_params: p.requires_grad_(False)   # freeze critics for actor step
            if algo == "SAC":
                a_pi, logp_pi = actor(s, deterministic=False, with_logprob=True)
                q_pi = torch.min(q1(s, a_pi), q2(s, a_pi))
                pi_loss = (SAC_ALPHA * logp_pi - q_pi).mean()   # max [Q - alpha*logp]
            else:
                a_pi = actor(s)
                pi_loss = -q1(s, a_pi).mean()                    # deterministic policy grad
            pi_opt.zero_grad(set_to_none=True)
            pi_loss.backward()
            pi_opt.step()
            for p in q_params: p.requires_grad_(True)

            # ----- Polyak target updates (on the same schedule as the actor) -----
            polyak(q1_targ, q1)
            if twin: polyak(q2_targ, q2)
            if algo != "SAC":                                # SAC has no target actor
                polyak(actor_targ, actor)

    # ---- main loop ---------------------------------------------------------
    steps_log, curve = [], []
    obs, _ = env.reset(seed=seed)
    for step in range(1, total_steps + 1):
        a = select_action(obs, step)
        s2, r, term, trunc, _ = env.step(a)
        # 'done' for bootstrap should ignore time-limit truncation:
        done_bootstrap = term and not trunc
        buf.add(obs, a, r, s2, done_bootstrap)
        obs = s2
        if term or trunc:
            obs, _ = env.reset()

        if step >= START_STEPS and len(buf) >= BATCH_SIZE:
            update(step)

        if step % EVAL_EVERY == 0:
            ret = evaluate(actor, algo, act_limit)
            steps_log.append(step); curve.append(ret)
            if verbose:
                print(f"  [{algo} seed={seed}] step {step:6d} | eval return {ret:8.1f}")

    env.close()
    if verbose:
        print(f"  [{algo} seed={seed}] done in {time.time() - t0:5.1f}s")
    return np.array(steps_log), np.array(curve), actor


## Run all three algorithms across seeds

We train `DDPG`, `TD3`, `SAC` for each seed in `SEEDS` and record the evaluation
curves. Elapsed time is printed per algorithm. (On free Colab CPU this is the slow
part -- a few minutes total with the default config.)


In [ ]:
results = {}          # algo -> list of curves (one per seed)
steps_axis = None     # shared x-axis (env steps)
last_actor = {}       # algo -> last trained actor (for the GIF / inspection)

for algo in ["DDPG", "TD3", "SAC"]:
    print(f"\n=== {algo} " + "=" * 40)
    t_algo = time.time()
    curves = []
    for seed in SEEDS:
        steps_log, curve, actor = train(algo, seed)
        curves.append(curve)
        last_actor[algo] = actor
        steps_axis = steps_log
    results[algo] = np.array(curves)            # shape (n_seeds, n_evals)
    print(f"=== {algo} total wall-clock: {time.time() - t_algo:5.1f}s "
          f"(final mean eval return {results[algo][:, -1].mean():.1f})")


## Learning curves: return vs. env steps (mean +/- std over seeds)

Pendulum returns are **negative** (best is $0$). Watch the curves climb toward
$\approx -200$ or better. The expected qualitative story:

- **DDPG**: gets going but is the noisiest / least reliable -- a symptom of Q
  overestimation and sensitivity to the exploration noise.
- **TD3**: steadier and usually higher final return -- the three tricks curb the bias.
- **SAC**: typically the fastest and most stable climber thanks to entropy-driven
  exploration.

(With only 2 seeds the bands are wide; that itself is a lesson about RL variance.)


In [ ]:
plt.figure(figsize=(8, 5))
colors = {"DDPG": "tab:red", "TD3": "tab:blue", "SAC": "tab:green"}

for algo in ["DDPG", "TD3", "SAC"]:
    curves = results[algo]                 # (n_seeds, n_evals)
    mean = curves.mean(axis=0)
    std  = curves.std(axis=0)
    plt.plot(steps_axis, mean, label=algo, color=colors[algo], lw=2)
    plt.fill_between(steps_axis, mean - std, mean + std, color=colors[algo], alpha=0.2)

plt.axhline(-200, ls="--", c="gray", lw=1, label="approx. 'good' threshold (-200)")
plt.xlabel("environment steps")
plt.ylabel("evaluation return (mean +/- std over seeds)")
plt.title("DDPG vs TD3 vs SAC on Pendulum-v1")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Quick numeric summary
print("Final-eval mean return per algorithm:")
for algo in ["DDPG", "TD3", "SAC"]:
    c = results[algo][:, -1]
    print(f"  {algo:5s}: {c.mean():8.1f}  +/- {c.std():5.1f}")


## Watch the trained SAC agent

Deterministic rollout of the SAC policy we just trained. It should swing the pendulum
up and hold it near vertical.


In [ ]:
sac_actor = last_actor["SAC"]
_env = gym.make(ENV_ID, render_mode="rgb_array")
act_limit = float(_env.action_space.high[0])

def sac_greedy(obs):
    o = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    with torch.no_grad():
        a, _ = sac_actor(o, deterministic=True, with_logprob=False)
    return np.clip(a.cpu().numpy()[0], -act_limit, act_limit)

_frames = episode_frames(_env, sac_greedy, max_steps=200, seed=7)
_env.close()
print("rendered", len(_frames), "frames")
frames_to_gif_html(_frames, fps=30)


## Takeaways & things to try

**The DDPG -> TD3 -> SAC story in one paragraph.** DDPG is the continuous-action analog
of DQN: a deterministic actor trained by the deterministic policy gradient through a
learned $Q$. Its Achilles heel is **maximization bias / Q-overestimation** -- the single
critic is bootstrapped from a target that the actor is actively pushing toward its
*argmax*, so noise in $Q$ gets systematically amplified into an optimistic bias, which
makes training brittle. **TD3** attacks exactly this with three cheap tricks:
(1) **twin critics + min** (Clipped Double-Q) to counter overestimation,
(2) **delayed policy updates** so the critic is more accurate before the actor chases it,
and (3) **target-policy smoothing** (clipped noise on the target action) to stop the
actor from exploiting sharp, spurious $Q$ peaks. **SAC** keeps the twin-critic min but
swaps the deterministic actor for a **stochastic squashed-Gaussian** one and adds an
**entropy bonus** to both the critic target and the actor objective. That entropy term
gives principled, state-dependent exploration and a smoother optimization landscape --
which is why SAC is usually the most sample-efficient and robust of the three.

**Things to try**
- **Ablate TD3 one trick at a time.** Set `POLICY_DELAY = 1` (no delay), or use a single
  critic instead of the `min` (delete the twin), or set `TARGET_NOISE = 0` (no smoothing).
  Watch which removal hurts most -- usually the twin-critic min.
- **Vary SAC's temperature `SAC_ALPHA`** (e.g. 0.05, 0.2, 0.5). Too small -> behaves like
  TD3 / can collapse exploration; too large -> the agent prefers wiggling over solving.
  Then implement **automatic temperature tuning** (learn $\alpha$ to hit a target
  entropy $\approx -\dim(a)$).
- **Look at the Q-values.** Log $Q(s,a)$ on eval states and compare them to actual
  discounted returns. DDPG's estimates should sit visibly *above* the truth
  (overestimation); TD3/SAC's should track it more closely. This makes the central
  pedagogical point quantitative.
- **More seeds / more steps.** RL is high-variance; rerun with `SEEDS=[0,1,2,3]` and a
  larger `TOTAL_STEPS` for tighter, more trustworthy bands.
- **Harder envs.** Swap `ENV_ID` for `"MountainCarContinuous-v0"` (sparse-ish reward,
  where exploration really matters) or, with a GPU, a MuJoCo task.

**Companion notes (Notion):**
- DDPG: https://app.notion.com/p/37f95008d76681079d8aeb200e134bfe
- TD3:  https://app.notion.com/p/37f95008d766814b84b9ea4942c3da4c
- SAC:  https://app.notion.com/p/37f95008d766811abd8de41c7e054dd2
